In [0]:
%run ./01_config

In [0]:
"""
08_figures.py  —  Dataset and cost figures

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 08 — Figures

# Renders inline and writes PNG (300 dpi) + SVG to figures/ in the volume, so the same
# run that produces a number produces the figure that shows it.

# Figure  -  Report use
# F1 order mix by class  -  Section 4.1.3 dataset statistics
# F2 Kaplan–Meier per class  -  Section 4.2.1 — model-free reference picture
# F3 censoring rate per class  -  Section 4.2.1 — why some classes fit poorly
# F4 cost distribution by order type  -  Section 4.1.1 cost calibration
# F5 incumbent cycle vs characteristic life  -  Section 4.3.1 — sets up the two-directional result
# F6 monthly notification volume  -  Section 5.1 — the temporal split
# F7 escalation rate by priority  -  Section 4.3.2 — is there signal for the prioritizer
# F8 interval durations by outcome  -  Section 4.2.1 — censoring structure
# F9 cost-rate curves  -  Section 4.3.1 — as-is vs optimum per class

# For quick throwaway exploration, Databricks' own chart builder is faster than any of
# this — display(spark.table(...)) then the chart icon under the result grid. Use that
# while poking around; use this notebook for anything going into the report.

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN, PURPLE = "#2b6cb0", "#c05621", "#2f855a", "#6b46c1"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})

dbutils.fs.mkdirs(FIGURES)

def emit(fig, name):
    """Save PNG + SVG to the volume, then render inline."""
    fig.tight_layout()
    stem = f"{FIGURES}/{DATASET_VERSION}_{name}"
    fig.savefig(f"{stem}.png", bbox_inches="tight")
    fig.savefig(f"{stem}.svg", bbox_inches="tight", format="svg")
    print(f"saved {stem}.png / .svg")
    plt.show()
    plt.close(fig)

# Load

iv = spark.table(tbl("gold_survival_intervals")).toPandas()
orders = spark.table(tbl("silver_order")).join(
    spark.table(tbl("silver_equipment")).select("equipment_id", "equipment_class"),
    "equipment_id", "left").toPandas()
notif = spark.table(tbl("silver_notification")).toPandas()
cycles = spark.table(tbl("gold_incumbent_cycles")).toPandas()

iv["interval_start"] = pd.to_datetime(iv.interval_start)
orders["basic_start"] = pd.to_datetime(orders.basic_start)
notif["notification_date"] = pd.to_datetime(notif.notification_date)

CUTOFF = pd.Timestamp(TRAIN_CUTOFF)
print(f"{len(iv):,} intervals | {len(orders):,} orders | {len(notif):,} notifications")

# F1 — Order mix by equipment class

p = (orders.pivot_table(index="equipment_class", columns="order_type",
                        values="order_id", aggfunc="count").fillna(0).sort_index(ascending=False))
labels = {"PM01": "PM01 breakdown", "PM02": "PM02 preventive", "PM03": "PM03 planned repair"}
colours = {"PM01": WARM, "PM02": ACCENT, "PM03": GREEN}

fig, ax = plt.subplots(figsize=(8, 4.4))
left = np.zeros(len(p))
for c in [c for c in ["PM01", "PM02", "PM03"] if c in p.columns]:
    ax.barh(p.index, p[c], left=left, color=colours[c], label=labels[c],
            height=.75, edgecolor="white", lw=.6)
    left += p[c].values
ax.set_xlabel("work orders")
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(.5, 1.01))
emit(fig, "f1_order_mix")

# F2 — Kaplan–Meier survival per class

# Model-free reference picture. Curves that stop early are classes whose intervals are
# almost all PM-censored — the shorter the incumbent cycle, the less of the lifetime
# distribution is ever observed.

def kaplan_meier(durations, events):
    df = pd.DataFrame({"t": durations, "e": events}).sort_values("t")
    n, s = len(df), 1.0
    ts, ss = [0.0], [1.0]
    for t, grp in df.groupby("t"):
        d = grp.e.sum()
        if d:
            s *= (1 - d / n)
        ts.append(t); ss.append(s); n -= len(grp)
    return np.array(ts), np.array(ss)

fig, ax = plt.subplots(figsize=(7.8, 4.6))
cmap = plt.get_cmap("tab10")
for i, (cls, g) in enumerate(iv.groupby("equipment_class")):
    t, s = kaplan_meier(g.duration_days.values, g.event_observed.values)
    ax.step(t, s, where="post", lw=1.4, color=cmap(i % 10), label=cls)
ax.set_xlabel("days since renewal"); ax.set_ylabel("S(t)"); ax.set_ylim(0, 1.02)
ax.legend(frameon=False, fontsize=7, ncol=2, loc="lower left")
emit(fig, "f2_kaplan_meier")

# F3 — Censoring rate and observed failures

# The diagnostic to read before trusting any per-class Weibull fit. A class at 90%+
# censoring is one where preventive maintenance has removed most of the failure evidence.

g = (iv.groupby("equipment_class")
       .agg(intervals=("duration_days", "size"), failures=("event_observed", "sum")))
g["censoring"] = 1 - g.failures / g.intervals
g = g.sort_values("censoring")

fig, ax = plt.subplots(figsize=(7.8, 4.2))
bars = ax.barh(g.index, g.censoring, color=[WARM if f < 30 else ACCENT for f in g.failures], height=.7)
for y, (c, f) in enumerate(zip(g.censoring, g.failures)):
    ax.text(c + .012, y, f"{c:.0%}   {f} failures", va="center", fontsize=7.5, color=MUTED)
ax.set_xlim(0, min(1.25, g.censoring.max() + .28)); ax.set_xlabel("censoring rate")
ax.set_title("orange = fewer than 30 observed failures", fontsize=8, color=MUTED, loc="left", pad=8)
emit(fig, "f3_censoring")

# F4 — Settled cost by order type

present = [t for t in ["PM02", "PM03", "PM01"] if t in orders.order_type.unique()]
data = [orders.loc[orders.order_type == t, "total_actual_cost"].dropna().values for t in present]

fig, ax = plt.subplots(figsize=(7.2, 3.8))
bp = ax.boxplot(data, vert=False, patch_artist=True, widths=.55, showfliers=False)
for patch, t in zip(bp["boxes"], present):
    patch.set_facecolor(colours[t]); patch.set_alpha(.75); patch.set_edgecolor(colours[t])
for m in bp["medians"]:
    m.set_color(INK); m.set_linewidth(1.4)
ax.set_yticks(range(1, len(present) + 1))
ax.set_yticklabels([labels[t] for t in present])
ax.set_xlabel("settled order cost (excludes downtime valuation)")
med = orders.groupby("order_type").total_actual_cost.median()
if {"PM01", "PM02"} <= set(med.index):
    ax.set_title(f"breakdown : preventive = {med['PM01'] / med['PM02']:.2f}x",
                 fontsize=8, color=MUTED, loc="left", pad=8)
emit(fig, "f4_cost_by_order_type")

# F5 — Incumbent cycle vs observed median time-to-failure

# A first look at where the optimizer is likely to recommend extending versus shortening.
# Classes whose cycle sits far below observed failure times are being over-serviced.

obs = (iv[iv.event_observed == 1].groupby("equipment_class").duration_days.median()
         .rename("median_ttf").reset_index())
comp = cycles.merge(obs, on="equipment_class", how="left").sort_values("cycle_days_mean")

y = np.arange(len(comp))
fig, ax = plt.subplots(figsize=(7.8, 4.4))
ax.hlines(y, comp.cycle_days_mean, comp.median_ttf, color=GRID, lw=2, zorder=1)
ax.scatter(comp.cycle_days_mean, y, color=PURPLE, s=45, zorder=2, label="incumbent PM cycle")
ax.scatter(comp.median_ttf, y, color=WARM, s=45, zorder=2, label="observed median time-to-failure")
ax.set_yticks(y); ax.set_yticklabels(comp.equipment_class)
ax.set_xlabel("days"); ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(.5, 1.01))
emit(fig, "f5_cycle_vs_ttf")

# F6 — Notification volume and the temporal split

m = notif.set_index("notification_date").resample("MS").size()

fig, ax = plt.subplots(figsize=(8.2, 3.4))
ax.fill_between(m.index, m.values, color=ACCENT, alpha=.22)
ax.plot(m.index, m.values, color=ACCENT, lw=1.3)
ax.axvline(CUTOFF, color=WARM, ls="--", lw=1.2)
ax.text(CUTOFF, m.max() * .97, "  train | test", color=WARM, fontsize=8, va="top")
ax.set_ylabel("notifications per month"); ax.set_xlabel("")
emit(fig, "f6_notification_volume")

# F7 — Escalation rate by priority code

# If the prioritizer is to beat priority-code ordering, escalation has to vary with
# something observable. Flat bars here would mean RQ3 has no signal to find.

if notif.originating_notification_id.notna().any():
    defects = notif[notif.notification_type == "M2"].copy()
    escalated_ids = set(notif.originating_notification_id.dropna())
    defects["escalated"] = defects.notification_id.isin(escalated_ids)
    r = defects.groupby("priority").escalated.agg(["mean", "size"])

    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    ax.bar(r.index.astype(str), r["mean"], color=WARM, width=.6)
    for x, (v, n) in enumerate(zip(r["mean"], r["size"])):
        ax.text(x, v + r["mean"].max() * .03, f"{v:.0%}\nn={n:,}",
                ha="center", fontsize=7.5, color=MUTED)
    ax.set_xlabel("notification priority (1 = most urgent)"); ax.set_ylabel("escalation rate")
    ax.set_ylim(0, r["mean"].max() * 1.4)
    emit(fig, "f7_escalation_by_priority")
else:
    print("Skipped: no escalation linkage in this dataset version (v2). Requires v2.1.")

# F8 — Interval durations by outcome

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.hist([iv.loc[iv.event_observed == 1, "duration_days"],
         iv.loc[iv.event_observed == 0, "duration_days"]],
        bins=40, stacked=True, color=[WARM, ACCENT],
        label=["failure", "right-censored"], edgecolor="white", lw=.3)
ax.set_xlabel("interval duration (days)"); ax.set_ylabel("intervals")
ax.legend(frameon=False)
emit(fig, "f8_interval_durations")

# F9 — Cost-rate curves: as-is vs optimum

# Uses the disclosed ground truth, so this is the true optimum, not a fitted estimate —
# the benchmark the fitted optimizer of Section 4.3.1 will be scored against. Run it before
# the modelling to see how much headroom the incumbent policy actually leaves.

if spark.catalog.tableExists(tbl("bronze_ground_truth_params")):
    gt = spark.table(tbl("bronze_ground_truth_params")).toPandas()
    for c in ["weibull_beta", "weibull_eta_days", "pm_cycle_days", "base_preventive_cost",
              "breakdown_cost_multiplier"]:
        gt[c] = gt[c].astype(float)

    def cost_rate(T, beta, eta, cp, cf):
        surv = lambda t: np.exp(-(t / eta) ** beta)
        grid = np.linspace(0, T, 600)
        expected_cycle = np.trapezoid(surv(grid), grid)
        return (cp * surv(T) + cf * (1 - surv(T))) / max(expected_cycle, 1e-9)

    rows, curves = [], {}
    for r in gt.itertuples():
        cp, cf = r.base_preventive_cost, r.base_preventive_cost * r.breakdown_cost_multiplier
        Ts = np.linspace(30, r.weibull_eta_days * 2.5, 400)
        rates = np.array([cost_rate(t, r.weibull_beta, r.weibull_eta_days, cp, cf) for t in Ts])
        as_is = cost_rate(r.pm_cycle_days, r.weibull_beta, r.weibull_eta_days, cp, cf)
        curves[r.EQTYP] = (Ts, rates / as_is, r.pm_cycle_days, Ts[rates.argmin()])
        rows.append({"equipment_class": r.EQTYP, "as_is_cycle": int(r.pm_cycle_days),
                     "true_optimum": int(round(Ts[rates.argmin()])),
                     "direction": "extend" if Ts[rates.argmin()] > r.pm_cycle_days else "shorten",
                     "cost_rate_saving_pct": round(100 * (as_is - rates.min()) / as_is, 1)})

    headroom = pd.DataFrame(rows).sort_values("cost_rate_saving_pct", ascending=False)
    display(spark.createDataFrame(headroom))

    n = len(curves)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(12, 5.6), sharey=True)
    for ax, (cls, (Ts, rel, as_is, opt)) in zip(axes.ravel(), curves.items()):
        ax.plot(Ts, rel, color=ACCENT, lw=1.5)
        ax.axvline(as_is, color=MUTED, ls="--", lw=1)
        ax.axvline(opt, color=WARM, lw=1.2)
        ax.set_title(cls.replace("-", "\n"), fontsize=7.5, color=INK)
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    axes[0, 0].set_ylabel("cost rate (as-is = 1.0)")
    fig.suptitle("grey dashed = incumbent cycle    orange = true cost optimum",
                 fontsize=8, color=MUTED, y=1.02)
    emit(fig, "f9_cost_rate_curves")
else:
    print("Skipped: bronze_ground_truth_params not loaded. Run 04_bronze_ingest with "
          "simulation_parameters.csv present in the ground_truth folder.")

display(dbutils.fs.ls(FIGURES))